# Queries Module - Fresh Data Queries

Refresh live data in Module 3. Market prices use MODULE_1_INPUT data.

```python
%run queries_module.ipynb

df_stocks = get_current_stocks()
df_prices = get_current_prices()
df_wac = get_current_wac()
df_cart = get_current_cart_rules()
```


In [ ]:
# =============================================================================
# IMPORTS & SNOWFLAKE CONNECTION
# =============================================================================
import pandas as pd
import os
import sys

# Add parent directory to path to import setup_environment_2
sys.path.append('..')
import setup_environment_2

# Initialize environment variables (loads Snowflake credentials)
setup_environment_2.initialize_env()

from db import query_snowflake, get_snowflake_timezone

TIMEZONE = get_snowflake_timezone()
print(f"Queries Module | Timezone: {TIMEZONE}")


In [ ]:
# =============================================================================
# QUERY 1: CURRENT STOCKS (from data_extraction.ipynb)
# =============================================================================
STOCK_QUERY = '''
with parent_whs as (
select * 
from (
VALUES
(236,343),
(1,467),
(962,343)
)x(parent_id,child_id)

)
select warehouse_id,product_id,case when stocks_child is not null and stocks = 0 then stocks_child else stocks end as stocks 
from (
SELECT 
    pw.warehouse_id,
    pw.product_id,
    pw.available_stock::INTEGER AS stocks,
	pw2.available_stock::INTEGER AS stocks_child
	

FROM product_warehouse pw
left join parent_whs p on p.parent_id = pw.warehouse_id
left join product_warehouse pw2 on pw2.warehouse_id = p.child_id and pw.product_id = pw2.product_id
WHERE pw.warehouse_id NOT IN (6, 9, 10)
    AND pw.is_basic_unit = 1
)
'''

def get_current_stocks():
    """Get fresh stock levels."""
    print("Fetching current stocks...")
    df = query_snowflake(STOCK_QUERY)
    print(f"  Loaded {len(df)} records")
    return df


In [ ]:
query = f'''
SELECT DISTINCT 
    qd.id AS discount_id,
    qd.dynamic_tag_id AS tag_id,
    qd.start_at,
    qd.end_at
FROM quantity_discounts qd 
WHERE qd.active = TRUE
    AND CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP()) 
        BETWEEN qd.start_at AND qd.end_at
'''
def get_active_qd_now():
    """Get fresh stock levels."""
    print("Fetching  qd ...")
    df = query_snowflake(query)
    print(f"  Loaded {len(df)} records")
    return df

In [ ]:
# =============================================================================
# QUERY 2: packing units
# =============================================================================
packing_units_QUERY = '''
with sales_check as (


SELECT  DISTINCT
		pso.product_id,
		pso.PACKING_UNIT_id,
        sum(pso.total_price) as nmv

FROM product_sales_order pso
JOIN sales_orders so ON so.id = pso.sales_order_id
             

WHERE so.created_at >= current_date - 60 
    AND so.sales_order_status_id not in (7,12)
    AND so.channel IN ('telesales','retailer')
    AND pso.purchased_item_count <> 0


GROUP BY 1,2
)
select product_id,packing_unit_id,basic_unit_count
from (
select *,max(nmv)over(partition by product_id,is_basic_unit) as top_nmv
from (
select 
pup.product_id,
pup.PACKING_UNIT_id,
pup.basic_unit_count,
pup.is_basic_unit,
count(distinct case when pup.basic_unit_count = 1 then pup.PACKING_UNIT_id end) over(partition by pup.product_id) as total_basic,
nmv
from PACKING_UNIT_PRODUCTS pup
left join sales_check sc on pup.product_id =sc.product_id and pup.PACKING_UNIT_id = sc.PACKING_UNIT_id
where pup.deleted_at is null
)

qualify case when total_basic > 1 then  nmv = top_nmv else true end
)
'''

def get_packing_units():
    """Get fresh stock levels."""
    print("Fetching packing_units ...")
    df = query_snowflake(packing_units_QUERY)
    print(f"  Loaded {len(df)} records")
    return df


In [ ]:
# =============================================================================
# QUERY 2: CURRENT PRICES (from data_extraction.ipynb)
# =============================================================================
CURRENT_PRICES_QUERY = f'''
WITH skus_prices AS (
    WITH local_prices AS (
        SELECT  
            CASE 
                WHEN cpu.cohort_id IN (700, 695) THEN 'Cairo'
                WHEN cpu.cohort_id IN (701) THEN 'Giza'
                WHEN cpu.cohort_id IN (704, 698) THEN 'Delta East'
                WHEN cpu.cohort_id IN (703, 697) THEN 'Delta West'
                WHEN cpu.cohort_id IN (696, 1123, 1124, 1125, 1126) THEN 'Upper Egypt'
                WHEN cpu.cohort_id IN (702, 699) THEN 'Alexandria'
            END AS region,
            cohort_id,
            pu.product_id,
            pu.packing_unit_id,
            pu.basic_unit_count,
            AVG(cpu.price) AS price
        FROM cohort_product_packing_units cpu
        JOIN PACKING_UNIT_PRODUCTS pu ON pu.id = cpu.product_packing_unit_id
        WHERE cpu.cohort_id IN (700,701,702,703,704,695,696,697,698,699,1123,1124,1125,1126)
            AND cpu.created_at::date <> '2023-07-31'
            AND cpu.is_customized = TRUE
        GROUP BY ALL
    ),
    
    live_prices AS (
        SELECT 
            region, cohort_id, product_id, 
            pu_id AS packing_unit_id, 
            buc AS basic_unit_count, 
            NEW_PRICE AS price
        FROM materialized_views.DBDP_PRICES
        WHERE created_at = CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::date
            AND DATE_PART('hour', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::time) 
                BETWEEN SPLIT_PART(time_slot, '-', 1)::int AND (SPLIT_PART(time_slot, '-', 1)::int) + 1
            AND cohort_id IN (700,701,702,703,704,695,696,697,698,699,1123,1124,1125,1126)
    ),
    
    prices AS (
        SELECT *
        FROM (
            SELECT *, 1 AS priority FROM live_prices
            UNION ALL
            SELECT *, 2 AS priority FROM local_prices
        )
        QUALIFY ROW_NUMBER() OVER (PARTITION BY region, cohort_id, product_id, packing_unit_id ORDER BY priority) = 1
    )
    
    SELECT region, cohort_id, product_id, price
    FROM prices
    WHERE basic_unit_count = 1
        AND ((product_id = 1309 AND packing_unit_id = 2) OR (product_id <> 1309))
)

SELECT distinct region, cohort_id, product_id, price as current_price
FROM skus_prices
'''

def get_current_prices():
    """Get fresh current prices."""
    print("Fetching current prices...")
    df = query_snowflake(CURRENT_PRICES_QUERY)
    print(f"  Loaded {len(df)} records")
    return df


In [ ]:
# =============================================================================
# QUERY: PERCENTILE DATA FOR CART RULES
# =============================================================================
PERCENTILE_QUERY = """
SELECT *,
    perc_95 + GREATEST(std_dev, 1) AS layer_1,
    perc_95 + (2 * GREATEST(std_dev, 1)) AS layer_2,
    perc_95 + (3 * GREATEST(std_dev, 1)) AS layer_3
FROM (
    SELECT cohort_id, product_id, sku, brand, cat,
           PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY purchased_item_count) as perc_25,
           PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY purchased_item_count) as perc_50,
           PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY purchased_item_count) as perc_75,
           PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY purchased_item_count) as perc_95,
           GREATEST(STDDEV(purchased_item_count), 1) as std_dev
    FROM (
        SELECT *, count(distinct order_id) over (partition by retailer_id, product_id) as rets_order
        FROM (
            SELECT cpc.cohort_id, pso.product_id,
                   CONCAT(p.name_ar,' ',p.size,' ',product_units.name_ar) as sku,
                   b.name_ar as brand, c.name_ar as cat,
                   so.id as order_id, so.retailer_id,
                   purchased_item_count * pso.basic_unit_count as purchased_item_count,
                   pso.total_price as nmv
            FROM sales_orders so  
            JOIN product_sales_order pso on pso.sales_order_id = so.id
            JOIN COHORT_PRICING_CHANGES cpc on cpc.id = pso.COHORT_PRICING_CHANGE_ID
            JOIN products p on p.id = pso.product_id 
            JOIN categories c on c.id = p.category_id
            JOIN brands b on b.id = p.brand_id
            JOIN product_units ON product_units.id = p.unit_id 
            WHERE so.created_at::Date >= date_trunc('month', current_date - 120)
            AND so.created_at::Date < current_date
            AND sales_order_status_id not in (7,12)
            AND cpc.cohort_id in (700,701,702,703,704,1123,1124,1125,1126)
        )
        QUALIFY rets_order >= 2 
    )
    GROUP BY ALL
)
"""

def get_percentile_data():
    """Get percentile data for cart rules (percentiles + layer tiers)."""
    print("Fetching percentile data for cart rules...")
    df = query_snowflake(PERCENTILE_QUERY)
    print(f"  Loaded {len(df)} percentile records")
    if len(df) > 0:
        print(f"   Percentiles available for {df['product_id'].nunique()} unique products")
    return df


In [ ]:
# =============================================================================
# QUERY 3: CURRENT WAC (from data_extraction.ipynb)
# =============================================================================
WAC_QUERY = f'''
SELECT 
    product_id,
    wac_p
FROM finance.all_cogs
WHERE CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP()) 
    BETWEEN from_date AND to_date
'''

def get_current_wac():
    """Get fresh WAC (Weighted Average Cost)."""
    print("Fetching current WAC...")
    df = query_snowflake(WAC_QUERY)
    print(f"  Loaded {len(df)} records")
    return df


In [ ]:
# =============================================================================
# QUERY 4: CURRENT CART RULES (from data_extraction.ipynb)
# =============================================================================
from constants import COHORT_IDS
# COHORT_IDS = [700, 701, 702, 703, 704, 1123, 1124, 1125, 1126]  # see constants.py

CART_RULES_QUERY = f'''
SELECT 
    cppu.cohort_id,
    pup.product_id,
    pup.basic_unit_count,
    COALESCE(cppu.MAX_PER_SALES_ORDER, cppu2.MAX_PER_SALES_ORDER) AS current_cart_rule
FROM COHORT_PRODUCT_PACKING_UNITS cppu 
JOIN PACKING_UNIT_PRODUCTS pup ON cppu.PRODUCT_PACKING_UNIT_ID = pup.id 
JOIN cohorts c ON c.id = cppu.cohort_id
LEFT JOIN COHORT_PRODUCT_PACKING_UNITS cppu2 
    ON cppu.PRODUCT_PACKING_UNIT_ID = cppu2.PRODUCT_PACKING_UNIT_ID 
    AND cppu2.cohort_id = c.FALLBACK_COHORT_ID
WHERE cppu.cohort_id IN ({",".join(map(str, COHORT_IDS))})
    AND pup.basic_unit_count = 1
'''

def get_current_cart_rules():
    """Get fresh cart rules."""
    print("Fetching current cart rules...")
    df = query_snowflake(CART_RULES_QUERY)
    df = df.groupby(['cohort_id', 'product_id']).agg(
        current_cart_rule=('current_cart_rule', 'min')
    ).reset_index()
    print(f"  Loaded {len(df)} records")
    return df


In [ ]:
# =============================================================================
# UTH (UP-TILL-HOUR) PERFORMANCE QUERIES
# =============================================================================
# Reusable queries for Module 3 and Module 4

def get_uth_performance():
    """
    Get today's Up-Till-Hour performance (qty and retailers from start of day to current hour).
    Uses Snowflake.
    Returns DataFrame with: warehouse_id, product_id, uth_qty, uth_nmv, uth_retailers
    """
    UTH_QUERY = f'''
    WITH parent_whs AS (
        SELECT * FROM (VALUES (236,343),(1,467),(962,343)) x(parent_id,child_id)
    ),
    params AS (
        SELECT
            CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE AS today,
            HOUR(CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())) AS current_hour
    ),
    sales_today AS (
        SELECT
            COALESCE(pw.parent_id, pso.warehouse_id) AS warehouse_id,
            pso.product_id,
            SUM(pso.purchased_item_count) AS qty,
            SUM(pso.total_price) AS nmv,
            so.retailer_id
        FROM product_sales_order pso
        LEFT JOIN parent_whs pw ON pw.child_id = pso.warehouse_id
        JOIN sales_orders so ON so.id = pso.sales_order_id
        CROSS JOIN params p
        WHERE so.created_at::date = p.today
            AND HOUR(so.created_at) < p.current_hour
            AND so.sales_order_status_id NOT IN (7, 12)
            AND so.channel IN ('telesales', 'retailer')
            AND pso.purchased_item_count <> 0
        GROUP BY COALESCE(pw.parent_id, pso.warehouse_id), pso.product_id, so.retailer_id
    )
    SELECT
        warehouse_id,
        product_id,
        SUM(qty) AS uth_qty,
        SUM(nmv) AS uth_nmv,
        COUNT(DISTINCT retailer_id) AS uth_retailers
    FROM sales_today
    GROUP BY warehouse_id, product_id
    '''
    print("Fetching UTH performance from Snowflake...")
    df = query_snowflake(UTH_QUERY)
    print(f"  Loaded {len(df)} UTH records")
    return df


def get_hourly_distribution():
    """
    Get historical hourly distribution (last 4 months) by category and warehouse.
    Uses Snowflake.
    Returns: warehouse_id, cat, avg_uth_pct_qty, avg_uth_pct_retailers, avg_last_hour_pct_qty, avg_last_hour_pct_retailers
    Note: Converts decimal.Decimal to float for compatibility.
    """
    HOURLY_DIST_QUERY = f'''
    WITH parent_whs AS (
        SELECT * FROM (VALUES (236,343),(1,467),(962,343)) x(parent_id,child_id)
    ),
    params AS (
        SELECT
            CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE AS today,
            CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 120 AS history_start,
            HOUR(CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())) AS current_hour,
            HOUR(CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())) - 1 AS last_hour
    ),
    hourly_sales AS (
        SELECT
            COALESCE(pw.parent_id, pso.warehouse_id) AS warehouse_id,
            c.name_ar AS cat,
            so.created_at::date AS sale_date,
            HOUR(so.created_at) AS sale_hour,
            SUM(pso.purchased_item_count) AS qty,
            COUNT(DISTINCT so.retailer_id) AS retailers
        FROM product_sales_order pso
        LEFT JOIN parent_whs pw ON pw.child_id = pso.warehouse_id
        JOIN sales_orders so ON so.id = pso.sales_order_id
        JOIN products p ON p.id = pso.product_id
        JOIN categories c ON c.id = p.category_id
        CROSS JOIN params
        WHERE so.created_at::date BETWEEN params.history_start AND params.today - 1
            AND so.sales_order_status_id NOT IN (7, 12)
            AND so.channel IN ('telesales', 'retailer')
            AND pso.purchased_item_count <> 0
        GROUP BY COALESCE(pw.parent_id, pso.warehouse_id), c.name_ar, so.created_at::date, sale_hour
    ),
    daily_totals AS (
        SELECT warehouse_id, cat, sale_date,
            SUM(qty) AS day_total_qty,
            SUM(retailers) AS day_total_retailers
        FROM hourly_sales
        GROUP BY warehouse_id, cat, sale_date
    ),
    uth_totals AS (
        SELECT hs.warehouse_id, hs.cat, hs.sale_date,
            SUM(hs.qty) AS uth_total_qty,
            SUM(hs.retailers) AS uth_total_retailers
        FROM hourly_sales hs
        CROSS JOIN params p
        WHERE hs.sale_hour < p.current_hour
        GROUP BY hs.warehouse_id, hs.cat, hs.sale_date
    ),
    last_hour_totals AS (
        SELECT hs.warehouse_id, hs.cat, hs.sale_date,
            SUM(hs.qty) AS last_hour_total_qty,
            SUM(hs.retailers) AS last_hour_total_retailers
        FROM hourly_sales hs
        CROSS JOIN params p
        WHERE hs.sale_hour = p.last_hour
        GROUP BY hs.warehouse_id, hs.cat, hs.sale_date
    )
    SELECT
        dt.warehouse_id, dt.cat,
        AVG(COALESCE(ut.uth_total_qty, 0) / NULLIF(dt.day_total_qty, 0)) AS avg_uth_pct_qty,
        AVG(COALESCE(ut.uth_total_retailers, 0) / NULLIF(dt.day_total_retailers, 0)) AS avg_uth_pct_retailers,
        AVG(COALESCE(lh.last_hour_total_qty, 0) / NULLIF(dt.day_total_qty, 0)) AS avg_last_hour_pct_qty,
        AVG(COALESCE(lh.last_hour_total_retailers, 0) / NULLIF(dt.day_total_retailers, 0)) AS avg_last_hour_pct_retailers
    FROM daily_totals dt
    LEFT JOIN uth_totals ut ON dt.warehouse_id = ut.warehouse_id 
        AND dt.cat = ut.cat AND dt.sale_date = ut.sale_date
    LEFT JOIN last_hour_totals lh ON dt.warehouse_id = lh.warehouse_id 
        AND dt.cat = lh.cat AND dt.sale_date = lh.sale_date
    WHERE dt.day_total_qty > 0
    GROUP BY dt.warehouse_id, dt.cat
    '''
    print("Fetching hourly distribution from Snowflake...")
    df = query_snowflake(HOURLY_DIST_QUERY)
    # Convert decimal.Decimal to float for compatibility
    for col in ['avg_uth_pct_qty', 'avg_uth_pct_retailers', 'avg_last_hour_pct_qty', 'avg_last_hour_pct_retailers']:
        if col in df.columns:
            df[col] = df[col].astype(float)
    print(f"  Loaded {len(df)} hourly distribution records")
    return df


def get_hourly_contribution_by_hour():
    """
    Get per-hour incremental contribution (0..23) by warehouse and category.
    Returns: warehouse_id, cat, hour, pct_contribution_qty, pct_contribution_retailers
    Used for in-stock hours weighting in Module 3 UTH target.
    """
    CONTRIB_BY_HOUR_QUERY = f'''
    WITH parent_whs AS (
        SELECT * FROM (VALUES (236,343),(1,467),(962,343)) x(parent_id,child_id)
    ),
    params AS (
        SELECT CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 120 AS history_start,
               CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 1 AS history_end
    ),
        hourly_sales AS (
        SELECT COALESCE(pw.parent_id, pso.warehouse_id) AS warehouse_id, c.name_ar AS cat, so.created_at::date AS sale_date,
               HOUR(so.created_at) AS sale_hour,
               SUM(pso.purchased_item_count*pso.basic_unit_count) AS qty,
               COUNT(DISTINCT so.retailer_id) AS retailers
        FROM product_sales_order pso
        LEFT JOIN parent_whs pw ON pw.child_id = pso.warehouse_id
        JOIN sales_orders so ON so.id = pso.sales_order_id
        JOIN products p ON p.id = pso.product_id
        JOIN categories c ON c.id = p.category_id
        CROSS JOIN params
        WHERE so.created_at::date BETWEEN params.history_start AND params.history_end
          AND so.sales_order_status_id NOT IN (7, 12)
          AND so.channel IN ('telesales', 'retailer')
          AND pso.purchased_item_count <> 0
        GROUP BY COALESCE(pw.parent_id, pso.warehouse_id), c.name_ar, so.created_at::date, sale_hour
    ),
    daily_totals AS (
        SELECT warehouse_id, cat, sale_date,
               SUM(qty) AS day_total_qty,
               SUM(retailers) AS day_total_retailers
        FROM hourly_sales
        GROUP BY warehouse_id, cat, sale_date
    )
	select warehouse_id,cat,hour,
	pct_contribution_qty/sum(pct_contribution_qty) over(partition by warehouse_id,cat) as pct_contribution_qty,
	pct_contribution_retailers/sum(pct_contribution_retailers) over(partition by warehouse_id,cat) as pct_contribution_retailers
	from (
    SELECT hs.warehouse_id, hs.cat, hs.sale_hour AS hour,
           AVG(hs.qty / NULLIF(dt.day_total_qty, 0)) AS pct_contribution_qty,
           AVG(hs.retailers / NULLIF(dt.day_total_retailers, 0)) AS pct_contribution_retailers
    FROM hourly_sales hs
    JOIN daily_totals dt ON hs.warehouse_id = dt.warehouse_id AND hs.cat = dt.cat AND hs.sale_date = dt.sale_date
    WHERE dt.day_total_qty > 0 
	GROUP BY hs.warehouse_id, hs.cat, hs.sale_hour
	)
    '''
    print("Fetching hourly contribution by hour from Snowflake...")
    df = query_snowflake(CONTRIB_BY_HOUR_QUERY)
    for col in ['pct_contribution_qty', 'pct_contribution_retailers']:
        if col in df.columns:
            df[col] = df[col].astype(float)
    print(f"  Loaded {len(df)} hourly contribution records")
    return df


def get_stock_snapshots_today():
    """
    Get hourly stock snapshots for today (Cairo date).
    Returns: product_id, warehouse_id, available_stock, hour
    Used for in-stock hours in Module 3 UTH target.
    """
    SNAPSHOTS_QUERY = f'''
    WITH parent_whs AS (
        SELECT * FROM (
            VALUES (236, 343), (1, 467), (962, 343)
        ) x(parent_id, child_id)
    )
    SELECT x.*,c.name_ar as cat
    FROM (
        SELECT sss.product_id, sss.warehouse_id, HOUR(sss.timestamp) AS hour,
               CASE WHEN sss2.available_stock IS NOT NULL AND sss.available_stock = 0 THEN sss2.available_stock ELSE sss.available_stock END AS available_stock
        FROM materialized_views.stock_snap_shots_recent sss
        JOIN packing_unit_products pup ON pup.product_id = sss.product_id AND pup.packing_unit_id = sss.packing_unit_id
        LEFT JOIN parent_whs p ON p.parent_id = sss.warehouse_id
        LEFT JOIN materialized_views.stock_snap_shots_recent sss2 ON sss2.warehouse_id = p.child_id AND sss.product_id = sss2.product_id AND sss.packing_unit_id = sss2.packing_unit_id AND sss.timestamp = sss2.timestamp
        WHERE sss.timestamp::date = CURRENT_DATE
        and pup.is_basic_unit = 1 
    )x
    join products p on p.id = x.product_id 
    join categories c on c.id = p.category_id
    WHERE x.available_stock > 0
    '''
    print("Fetching today's stock snapshots from Snowflake...")
    df = query_snowflake(SNAPSHOTS_QUERY)
    if 'available_stock' in df.columns:
        df['available_stock'] = pd.to_numeric(df['available_stock'], errors='coerce').fillna(0)
    print(f"  Loaded {len(df)} stock snapshot rows")
    return df


def get_last_hour_performance():
    """
    Get last hour performance from DWH (PostgreSQL).
    Returns DataFrame with: warehouse_id, product_id, last_hour_qty, last_hour_nmv, last_hour_retailers
    """
    LAST_HOUR_QUERY = f'''
WITH parent_whs AS (
    SELECT * FROM (VALUES (236,343),(1,467),(962,343)) AS x(parent_id,child_id)
)
SELECT
    COALESCE(pw.parent_id, pso.warehouse_id) AS warehouse_id,
    pso.product_id,
    SUM(pso.purchased_item_count) AS last_hour_qty,
    SUM(pso.total_price) AS last_hour_nmv,
    COUNT(DISTINCT so.retailer_id) AS last_hour_retailers
FROM product_sales_order pso
LEFT JOIN parent_whs pw ON pw.child_id = pso.warehouse_id
JOIN sales_orders so 
    ON so.id = pso.sales_order_id
WHERE so.created_at::date =
      (CURRENT_TIMESTAMP AT TIME ZONE 'Africa/Cairo')::date
AND EXTRACT(
        HOUR FROM so.created_at
    ) =
    EXTRACT(
        HOUR FROM CURRENT_TIMESTAMP AT TIME ZONE 'Africa/Cairo'
    ) - 1
AND so.sales_order_status_id NOT IN (7, 12)
AND so.channel IN ('telesales', 'retailer')
AND pso.purchased_item_count <> 0
GROUP BY COALESCE(pw.parent_id, pso.warehouse_id), pso.product_id;


    '''
    print("Fetching last hour performance from DWH...")
    df = setup_environment_2.dwh_pg_query(
        LAST_HOUR_QUERY, 
        columns=['warehouse_id', 'product_id', 'last_hour_qty', 'last_hour_nmv', 'last_hour_retailers']
    )
    df.columns = df.columns.str.lower()
    # Convert to numeric
    for col in ['warehouse_id', 'product_id', 'last_hour_qty', 'last_hour_nmv', 'last_hour_retailers']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    print(f"  Loaded {len(df)} last hour records from DWH")
    return df

print("✅ UTH and Last Hour functions defined")


In [ ]:
# =============================================================================
# YESTERDAY'S CLOSING STOCK (from stock_day_close with parent warehouse mapping)
# =============================================================================

def get_yesterday_closing_stock():
    """
    Get yesterday's closing stock from stock_day_close.
    Applies parent_warehouse mapping: if parent warehouse stock = 0, use child warehouse stock.
    Returns: product_id, warehouse_id, closing_stock_yesterday
    """
    CLOSING_STOCK_QUERY = f'''
    WITH parent_whs AS (
        SELECT * FROM (
            VALUES (236, 343), (1, 467), (962, 343)
        ) x(parent_id, child_id)
    )
    SELECT product_id, warehouse_id,
        CASE WHEN closing_child IS NOT NULL AND closing_stocks = 0 
             THEN closing_child ELSE closing_stocks END AS closing_stock_yesterday
    FROM (
        SELECT sdc.product_id, sdc.warehouse_id, 
               sdc.AVAILABLE_STOCK AS closing_stocks,
               sdc2.AVAILABLE_STOCK AS closing_child
        FROM materialized_views.stock_day_close sdc
        LEFT JOIN parent_whs p ON p.parent_id = sdc.warehouse_id
        LEFT JOIN materialized_views.stock_day_close sdc2 
            ON sdc2.warehouse_id = p.child_id 
            AND sdc.product_id = sdc2.product_id 
            AND sdc.timestamp = sdc2.timestamp
        WHERE sdc.timestamp::DATE = CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 1
    )
    '''
    print("Fetching yesterday's closing stock from Snowflake...")
    df = query_snowflake(CLOSING_STOCK_QUERY)
    for col in ['closing_stock_yesterday']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
    print(f"  Loaded {len(df)} closing stock records")
    return df

print("✅ Yesterday closing stock function defined")

In [ ]:
# =============================================================================
# FIXED PRICE OVERRIDE (from Google Sheet)
# =============================================================================

def get_fixed_prices():
    """
    Read the 'Fixed Price' Google Sheet.
    Returns: DataFrame with product_id, fixed_price
    """
    from common_functions import google_sheets
    print("Fetching fixed prices from Google Sheet...")
    df = google_sheets('Fixed Price', 'Sheet1', 'get')
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
    df['product_id'] = pd.to_numeric(df['product_id'], errors='coerce')
    df['market_price'] = pd.to_numeric(df['market_price'], errors='coerce')
    df = df.dropna(subset=['product_id', 'market_price'])
    df['product_id'] = df['product_id'].astype(int)
    result = df[['product_id', 'market_price']].rename(columns={'market_price': 'fixed_price'})
    result = result.drop_duplicates(subset='product_id', keep='first')
    print(f"  Loaded {len(result)} fixed price SKUs")
    return result


def get_fixed_cart_rules():
    """
    Read the 'Fixed Price' Google Sheet, tab 'Sheet2'.
    Returns: DataFrame with product_id, fixed_cart_rule
    """
    from common_functions import google_sheets
    print("Fetching fixed cart rules from Google Sheet...")
    df = google_sheets('Fixed Price', 'Sheet2', 'get')
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
    df['product_id'] = pd.to_numeric(df['product_id'], errors='coerce')
    df['cart_rule'] = pd.to_numeric(df['cart_rule'], errors='coerce')
    df = df.dropna(subset=['product_id', 'cart_rule'])
    df['product_id'] = df['product_id'].astype(int)
    df['cart_rule'] = df['cart_rule'].astype(int)
    result = df[['product_id', 'cart_rule']].rename(columns={'cart_rule': 'fixed_cart_rule'})
    result = result.drop_duplicates(subset='product_id', keep='first')
    print(f"  Loaded {len(result)} fixed cart rule SKUs")
    return result

print("Fixed price & cart rule functions defined")

In [ ]:
# =============================================================================
# COMMERCIAL MINIMUM PRICE CONSTRAINTS (from finance.minimum_prices)
# Same query as data_extraction.ipynb — fetched fresh by M3/M4 each run
# =============================================================================
COMMERCIAL_MIN_PRICE_QUERY = f'''
WITH to_remove AS (
    SELECT 
        check_date AS start_date,
        (check_date + INTERVAL '1 month') + 6 AS end_date 
    FROM (
        SELECT 
            CASE 
                WHEN DATE_PART('day', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE) < 7 
                THEN DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - INTERVAL '1 month') 
                ELSE DATE_FROM_PARTS(
                    YEAR(CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE), 
                    MONTH(CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE), 
                    1
                )  
            END AS check_date
    )
),
region_mapping AS (
    SELECT * FROM (VALUES
        ('Greater Cairo', 'Cairo'),
        ('Greater Cairo', 'Giza')
    ) x(region, main_region)
)
SELECT  
    sku_id AS product_id,
    coalesce(rm.main_region, mp.region) AS region,
    min_price AS commercial_min_price
FROM (
    SELECT 
        product_id AS sku_id,
        mp.region,
        min_price,
        created_at,
        MAX(created_at) OVER (PARTITION BY product_id, mp.region) AS latest_date
    FROM finance.minimum_prices mp
    WHERE is_deleted = 'false'
        AND created_at BETWEEN (SELECT start_date FROM to_remove) AND (SELECT end_date FROM to_remove)
) comm
LEFT JOIN region_mapping rm ON comm.region = rm.region
WHERE created_at = latest_date
'''

def get_commercial_min_prices():
    """Get fresh commercial minimum price constraints from finance.minimum_prices."""
    print("Fetching commercial min prices...")
    df = query_snowflake(COMMERCIAL_MIN_PRICE_QUERY)
    for col in ['product_id', 'commercial_min_price']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    print(f"  Loaded {len(df)} commercial min price records")
    return df[['product_id', 'region', 'commercial_min_price']]

print("get_commercial_min_prices() defined")

In [ ]:
# =============================================================================
# READY
# =============================================================================
print("\n" + "="*50)
print("QUERIES MODULE READY")
print("="*50)
print("\nLive Data Functions:")
print("  • get_current_stocks()")
print("  • get_packing_units()")
print("  • get_current_prices()")
print("  • get_current_wac()")
print("  • get_current_cart_rules()")
print("\nUTH Performance Functions:")
print("  • get_uth_performance()         - UTH qty/retailers (Snowflake)")
print("  • get_hourly_distribution()     - Historical hour contributions (Snowflake)")
print("  • get_last_hour_performance()   - Last hour qty/retailers (DWH)")
print("\nStock Functions:")
print("  • get_yesterday_closing_stock() - Yesterday closing stock with parent WH mapping")
print("\nOverride Functions:")
print("  • get_fixed_prices()            - Fixed prices from Google Sheet")
print("\nCommercial Constraints:")
print("  • get_commercial_min_prices()   - Fresh min price constraints from finance.minimum_prices")
print("\nNote: Market prices use MODULE_1_INPUT data")


In [ ]:
# =============================================================================
# QUARTERLY CONTRIBUTION FACTOR (qtr_cntrb)
# =============================================================================
# Recency-weighted (exponential) quarterly demand contribution per category.
# Used to scale P80 targets: peak quarters get boosted, weak quarters get reduced.
# Baseline = 2nd highest quarter (cntrb = 1.0), bounded to [0.9, 1.1].

QTR_CONTRIBUTION_QUERY = '''
SELECT cat, qtr_cntrb FROM (
    SELECT *,
        LEAST(GREATEST(
            weighted_avg_cntrb / NTH_VALUE(weighted_avg_cntrb, 2)
                OVER (PARTITION BY cat ORDER BY weighted_avg_cntrb DESC
                      ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING),
            0.9
        ), 1.1) AS qtr_cntrb
    FROM (
        SELECT qtr_num, cat,
            weighted_avg_cntrb / SUM(weighted_avg_cntrb) OVER (PARTITION BY cat) AS weighted_avg_cntrb
        FROM (
            SELECT qtr_num, cat,
                SUM(cntrb * year_weight) / SUM(year_weight) AS weighted_avg_cntrb
            FROM (
                SELECT *,
                    qty / SUM(qty) OVER (PARTITION BY qtr_year, cat) AS cntrb,
                    POWER(2, qtr_year - MIN(qtr_year) OVER ()) AS year_weight
                FROM (
                    SELECT DISTINCT
                        QUARTER(so.created_at::date) AS qtr_num,
                        YEAR(so.created_at::date) AS qtr_year,
                        categories.name_ar AS cat,
                        SUM(pso.purchased_item_count * pso.basic_unit_count) AS qty
                    FROM product_sales_order pso
                    JOIN sales_orders so ON so.id = pso.sales_order_id
                    JOIN products ON products.id = pso.product_id
                    JOIN categories ON products.category_id = categories.id
                    WHERE so.created_at::date BETWEEN DATE_TRUNC('year', CURRENT_DATE - INTERVAL '3 years')
                                                  AND DATE_TRUNC('year', CURRENT_DATE) - 1
                        AND so.sales_order_status_id NOT IN (7, 12)
                        AND so.channel IN ('telesales', 'retailer')
                        AND pso.purchased_item_count <> 0
                    GROUP BY ALL
                )
            )
            GROUP BY qtr_num, cat
        )
    )
) WHERE qtr_num = QUARTER(CURRENT_DATE)

'''

def get_quarterly_contribution():
    """Get quarterly contribution factor (qtr_cntrb) per category for the current quarter.
    Based on 3 years of recency-weighted (exponential) quarterly demand.
    Bounded to [0.9, 1.1], baseline is the 2nd highest quarter."""
    print("  Fetching quarterly contribution factors...")
    df = query_snowflake(QTR_CONTRIBUTION_QUERY)
    print(f"    Found qtr_cntrb for {len(df)} categories")
    return df

print("Quarterly Contribution Query defined ✓")
print("  - get_quarterly_contribution()")

In [ ]:
# =============================================================================
# TARGET TURNOVER QTY (for high-DOH SKUs)
# =============================================================================
# For SKUs with DOH > 15, calculates a daily target_qty to accelerate stock depletion.
# target_qty = avg_stock / target_DOH, where target_DOH = max(current_DOH * 0.85, 12).
# Only returns rows for SKUs that need stock depletion help.

TARGET_TURNOVER_QUERY = f'''
WITH parent_whs AS (
    SELECT * FROM (VALUES (236, 343), (1, 467), (962, 343)) x(parent_id, child_id)
),

current_stocks AS (
    SELECT 
        pw.warehouse_id, 
        pw.product_id,
        CASE 
            WHEN pw2.available_stock IS NOT NULL AND pw.available_stock = 0 
            THEN pw2.available_stock 
            ELSE pw.available_stock 
        END::INTEGER AS current_stock
    FROM product_warehouse pw
    LEFT JOIN parent_whs p ON p.parent_id = pw.warehouse_id
    LEFT JOIN product_warehouse pw2 
        ON pw2.warehouse_id = p.child_id 
        AND pw.product_id = pw2.product_id
    WHERE pw.warehouse_id NOT IN (6, 9, 10) 
      AND pw.is_basic_unit = 1
),

closing_stocks AS (
    SELECT distinct 
        sdc.product_id, 
        sdc.warehouse_id,
        CASE 
            WHEN sdc2.AVAILABLE_STOCK IS NOT NULL AND sdc.AVAILABLE_STOCK = 0 
            THEN sdc2.AVAILABLE_STOCK 
            ELSE sdc.AVAILABLE_STOCK 
        END::INTEGER AS closing_stock_yesterday
    FROM materialized_views.stock_day_close sdc
    LEFT JOIN parent_whs p ON p.parent_id = sdc.warehouse_id
    LEFT JOIN materialized_views.stock_day_close sdc2 
        ON sdc2.warehouse_id = p.child_id 
        AND sdc.product_id = sdc2.product_id 
        AND sdc.timestamp = sdc2.timestamp
    WHERE sdc.timestamp::DATE = CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 1
),

yesterday_sales AS (
    SELECT 
        COALESCE(pw.parent_id, pso.warehouse_id) AS warehouse_id, 
        pso.product_id,
        SUM(pso.purchased_item_count * pso.basic_unit_count) AS sold_qty_yesterday
    FROM product_sales_order pso
    LEFT JOIN parent_whs pw ON pw.child_id = pso.warehouse_id
    JOIN sales_orders so ON so.id = pso.sales_order_id
    WHERE so.created_at::DATE = CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 1
      AND so.sales_order_status_id NOT IN (7, 12)
      AND so.channel IN ('telesales', 'retailer')
      AND pso.purchased_item_count <> 0
    GROUP BY all
),
mtd_sales AS (
select warehouse_id,product_id,avg(qty) AS mtd_avg_qty

from (
    SELECT 
		so.created_at::date o_date,
        COALESCE(pw.parent_id, pso.warehouse_id) AS warehouse_id, 
        pso.product_id,
        SUM(pso.purchased_item_count * pso.basic_unit_count) AS qty
    FROM product_sales_order pso
    LEFT JOIN parent_whs pw ON pw.child_id = pso.warehouse_id
    JOIN sales_orders so ON so.id = pso.sales_order_id
    WHERE so.created_at::DATE between  date_trunc('month',CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE) and current_Date - 1 
      AND so.sales_order_status_id NOT IN (7, 12)
      AND so.channel IN ('telesales', 'retailer')
      AND pso.purchased_item_count <> 0
    GROUP BY all
	)
	group by all
)


select *, greatest(avg_stock/least(greatest(least(turnover_yesterday_doh*0.6,turnover_mtd_doh*0.85), 7),30),5) as target_qty
from (
SELECT distinct 
    cs.warehouse_id,
    cs.product_id,
	wac1,
    cs.current_stock,
    COALESCE(cls.closing_stock_yesterday, 0) AS closing_stock_yesterday,
    greatest(cs.current_stock , COALESCE(cls.closing_stock_yesterday, 0)) AS avg_stock,
    COALESCE(ys.sold_qty_yesterday, 0) AS sold_qty_yesterday,
	COALESCE(ms.mtd_avg_qty, 0) AS mtd_avg_qty,
    CASE 
        WHEN COALESCE(ys.sold_qty_yesterday, 0) > 0 
        THEN ROUND(
            avg_stock
            / ys.sold_qty_yesterday, 2)
        ELSE  least(avg_stock,100)
    END AS turnover_yesterday_doh,

    CASE 
        WHEN COALESCE(ms.mtd_avg_qty, 0) > 0 
        THEN ROUND(
            avg_stock
            / ms.mtd_avg_qty, 2)
        ELSE  least(avg_stock,100)
    END AS turnover_mtd_doh
	
FROM current_stocks cs
LEFT JOIN closing_stocks cls 
    ON cls.warehouse_id = cs.warehouse_id AND cls.product_id = cs.product_id
LEFT JOIN yesterday_sales ys 
    ON ys.warehouse_id = cs.warehouse_id AND ys.product_id = cs.product_id
LEFT JOIN mtd_sales ms 
    ON ms.warehouse_id = cs.warehouse_id AND ms.product_id = cs.product_id
	
join finance.all_cogs f on f.product_id = cs.product_id and CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP()) between from_Date and to_date 	
WHERE cs.current_stock > 0
and avg_stock * wac1 > 1000
and turnover_yesterday_doh > 8.2
and closing_stock_yesterday > 0
)
order by target_qty desc
'''

def get_target_turnover_qty():
    """Get target daily qty for high-DOH SKUs (DOH > 15) to accelerate stock depletion.
    Returns warehouse_id, product_id, target_qty. Only includes SKUs needing help."""
    print("  Fetching target turnover quantities...")
    df = query_snowflake(TARGET_TURNOVER_QUERY)
    print(f"    Found target_qty for {len(df)} high-DOH SKUs")
    return df

print("Target Turnover Query defined ✓")
print("  - get_target_turnover_qty()")

In [ ]:
# =============================================================================
# RETAILER SELECTION QUERIES (for SKU Discount Handler)
# =============================================================================

def get_churned_dropped_retailers(selected_skus_tuple: str) -> pd.DataFrame:
    """
    Query 1: Get retailers who were buying this product but dropped >30%.
    These are churned/dropping retailers who might respond to a discount.
    
    Args:
        selected_skus_tuple: String of tuples like "(1, 2), (3, 4)"
    
    Returns:
        DataFrame with retailer_id, product_id, warehouse_id
    """
    query = f'''
    WITH selected_prods AS (
        SELECT * 
        FROM (VALUES {selected_skus_tuple}) x(product_id, warehouse_id)
    ),
    retailer_current_wh AS (
        SELECT DISTINCT rp.retailer_id, sp.product_id, sp.warehouse_id
        FROM materialized_views.retailer_polygon rp 
        JOIN DISPATCHING_POLYGONS dp ON dp.district_id = rp.district_id
        JOIN WAREHOUSE_DISPATCHING_RULES wdr ON wdr.DISPATCHING_POLYGON_ID = dp.id
        JOIN selected_prods sp ON sp.product_id = wdr.product_id AND sp.warehouse_id = wdr.warehouse_id
    ),
    sales_before AS (
        SELECT retailer_id, product_id, avg(nmv) as avg_nmv_before
        FROM (
            SELECT DISTINCT
                so.id as order_id,
                pso.product_id as product_id,
                so.retailer_id as retailer_id,
                sum(pso.total_price) as nmv 
            FROM product_sales_order pso
            JOIN sales_orders so ON so.id = pso.sales_order_id
            JOIN selected_prods sp ON sp.product_id = pso.product_id
            WHERE so.created_at::date BETWEEN CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 120 
                  AND CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 31
                AND so.sales_order_status_id NOT IN (7, 12)
                AND so.channel IN ('telesales', 'retailer')
                AND pso.purchased_item_count <> 0
            GROUP BY ALL
        )
        GROUP BY ALL 
    ),
    sales_after AS (
        SELECT retailer_id, product_id, avg(nmv) as avg_nmv_after, max(order_date) as last_order
        FROM (
            SELECT DISTINCT
                so.id as order_id,
                so.created_at::date as order_date,
                pso.product_id as product_id,
                so.retailer_id as retailer_id,
                sum(pso.total_price) as nmv 
            FROM product_sales_order pso
            JOIN sales_orders so ON so.id = pso.sales_order_id
            JOIN selected_prods sp ON sp.product_id = pso.product_id
            WHERE so.created_at::date > CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 31
                AND so.sales_order_status_id NOT IN (7, 12)
                AND so.channel IN ('telesales', 'retailer')
                AND pso.purchased_item_count <> 0
            GROUP BY ALL
        )
        GROUP BY ALL 
    ),
    made_order AS (
        SELECT DISTINCT so.retailer_id
        FROM sales_orders so 
        JOIN product_sales_order pso ON pso.sales_order_id = so.id 
        WHERE so.created_at::date >= CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 60
            AND so.sales_order_status_id NOT IN (7, 12)
            AND so.channel IN ('telesales', 'retailer')
        GROUP BY ALL
    )
    SELECT DISTINCT rcw.retailer_id, rcw.product_id, rcw.warehouse_id
    FROM (
        SELECT sb.*, COALESCE(avg_nmv_after, 0) as nmv_after, 
               (nmv_after - avg_nmv_before) / avg_nmv_before as growth
        FROM sales_before sb 
        LEFT JOIN sales_after sa ON sb.retailer_id = sa.retailer_id AND sb.product_id = sa.product_id 
        LEFT JOIN made_order mo ON mo.retailer_id = sa.retailer_id 
        WHERE growth < -0.3
            AND (CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - last_order >= 5 OR last_order IS NULL)
            AND mo.retailer_id IS NOT NULL 
    ) churned
    JOIN retailer_current_wh rcw ON rcw.retailer_id = churned.retailer_id AND rcw.product_id = churned.product_id
    '''
    print("  Fetching churned/dropped retailers...")
    df = query_snowflake(query)
    print(f"    Found {len(df)} churned/dropped retailer-product combinations")
    return df


def get_category_not_product_retailers(selected_skus_tuple: str) -> pd.DataFrame:
    """
    Query 2: Get retailers who buy the category but not this specific product.
    These are potential new customers for the product.
    
    Args:
        selected_skus_tuple: String of tuples like "(1, 2), (3, 4)"
    
    Returns:
        DataFrame with retailer_id, product_id, warehouse_id
    """
    query = f'''
    WITH selected_prods AS (
    SELECT * 
    FROM (VALUES {selected_skus_tuple}) x(product_id, warehouse_id)
),

eligible_retailers AS (
    SELECT DISTINCT rp.retailer_id, sp.product_id, sp.warehouse_id
    FROM materialized_views.retailer_polygon rp 
    JOIN DISPATCHING_POLYGONS dp ON dp.district_id = rp.district_id
    JOIN WAREHOUSE_DISPATCHING_RULES wdr ON wdr.DISPATCHING_POLYGON_ID = dp.id
    JOIN selected_prods sp ON sp.product_id = wdr.product_id AND sp.warehouse_id = wdr.warehouse_id
),

selected_product_info AS (
    SELECT sp.product_id, c.name_ar AS cat, b.name_ar AS brand
    FROM selected_prods sp
    JOIN products p ON p.id = sp.product_id
    JOIN brands b ON b.id = p.brand_id 
    JOIN categories c ON c.id = p.category_id 
),

category_buyers AS (
    SELECT so.retailer_id, COUNT(DISTINCT so.id) AS order_count
    FROM product_sales_order pso
    JOIN sales_orders so ON so.id = pso.sales_order_id
    JOIN products p ON p.id = pso.product_id
    JOIN brands b ON b.id = p.brand_id 
    JOIN categories c ON c.id = p.category_id 
    JOIN selected_product_info si ON si.cat = c.name_ar AND si.brand = b.name_ar
    WHERE so.created_at::date >= CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 60
        AND so.sales_order_status_id NOT IN (7, 12)
        AND so.channel IN ('telesales', 'retailer')
        AND pso.purchased_item_count <> 0
    GROUP BY so.retailer_id
    HAVING COUNT(DISTINCT so.id) > 1
),

product_buyers AS (
    SELECT DISTINCT so.retailer_id
    FROM product_sales_order pso
    JOIN sales_orders so ON so.id = pso.sales_order_id
    JOIN selected_prods sp ON sp.product_id = pso.product_id
    WHERE so.created_at::date >= CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 60
        AND so.sales_order_status_id NOT IN (7, 12)
        AND so.channel IN ('telesales', 'retailer')
        AND pso.purchased_item_count <> 0
)

SELECT er.retailer_id, er.product_id, er.warehouse_id, cb.order_count
FROM eligible_retailers er
JOIN category_buyers cb ON cb.retailer_id = er.retailer_id
WHERE NOT EXISTS (
    SELECT 1 FROM product_buyers pb WHERE pb.retailer_id = er.retailer_id
)
ORDER BY cb.order_count DESC
    '''
    print("  Fetching category-not-product retailers...")
    df = query_snowflake(query)
    print(f"    Found {len(df)} category-not-product retailer-product combinations")
    return df


def get_out_of_cycle_retailers(selected_skus_tuple: str) -> pd.DataFrame:
    """
    Query 3: Get retailers who should have reordered by now based on their purchase cycle.
    
    Args:
        selected_skus_tuple: String of tuples like "(1, 2), (3, 4)"
    
    Returns:
        DataFrame with retailer_id, product_id, warehouse_id
    """
    query = f'''
    WITH selected_prods AS (
        SELECT * 
        FROM (VALUES {selected_skus_tuple}) x(product_id, warehouse_id)
    ),
    retailer_current_wh AS (
        SELECT DISTINCT rp.retailer_id, sp.product_id, sp.warehouse_id
        FROM materialized_views.retailer_polygon rp 
        JOIN DISPATCHING_POLYGONS dp ON dp.district_id = rp.district_id
        JOIN WAREHOUSE_DISPATCHING_RULES wdr ON wdr.DISPATCHING_POLYGON_ID = dp.id
        JOIN selected_prods sp ON sp.product_id = wdr.product_id AND sp.warehouse_id = wdr.warehouse_id
    ),
    out_of_cycle AS (
        SELECT retailer_id, product_id
        FROM (
            SELECT *, last_o_date + floor(avg_cycle + (2.5 * std))::int as next_order
            FROM (
                SELECT retailer_id, product_id, max(last_o_date) as last_o_date, 
                    sum(order_days * (w / all_w)) as avg_cycle, stddev(order_days) as std
                FROM (
                    SELECT *,
                        max(order_num) OVER(PARTITION BY retailer_id, product_id) as max_orders,
                        lag(o_date) OVER(PARTITION BY product_id, retailer_id ORDER BY o_date) as prev_order,
                        o_date - prev_order as order_days,
                        CASE WHEN CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - o_date = 0 
                             THEN 1 ELSE 1 / (CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - o_date) END as w,
                        sum(w) OVER(PARTITION BY product_id, retailer_id) as all_w
                    FROM (
                        SELECT DISTINCT
                            so.id as order_id,
                            so.created_at::date as o_date,
                            pso.product_id as product_id,
                            so.retailer_id as retailer_id,
                            sum(pso.total_price) as nmv,
                            row_number() OVER(PARTITION BY so.retailer_id, pso.product_id ORDER BY o_date DESC) as order_num,
                            max(o_date) OVER(PARTITION BY so.retailer_id, pso.product_id) as last_o_date
                        FROM product_sales_order pso
                        JOIN sales_orders so ON so.id = pso.sales_order_id
                        JOIN selected_prods sp ON sp.product_id = pso.product_id
                        WHERE so.created_at::date >= DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - INTERVAL '1 year')
                            AND so.sales_order_status_id NOT IN (7, 12)
                            AND so.channel IN ('telesales', 'retailer')
                            AND pso.purchased_item_count <> 0
                        GROUP BY 1, 2, 3, 4
                    )
                    WHERE last_o_date >= CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 60
                    QUALIFY max_orders >= 4
                )
                WHERE prev_order IS NOT NULL 
                GROUP BY ALL
            )
            WHERE CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE >= next_order
        )
    )
    SELECT DISTINCT rcw.retailer_id, rcw.product_id, rcw.warehouse_id
    FROM out_of_cycle ooc
    JOIN retailer_current_wh rcw ON rcw.retailer_id = ooc.retailer_id AND rcw.product_id = ooc.product_id
    '''
    print("  Fetching out-of-cycle retailers...")
    df = query_snowflake(query)
    print(f"    Found {len(df)} out-of-cycle retailer-product combinations")
    return df


def get_view_no_orders_retailers(selected_skus_tuple: str) -> pd.DataFrame:
    """
    Query 4: Get retailers who viewed the brand but didn't order.
    
    Args:
        selected_skus_tuple: String of tuples like "(1, 2), (3, 4)"
    
    Returns:
        DataFrame with retailer_id, product_id, warehouse_id
    """
    query = f'''
    WITH selected_prods AS (
    SELECT * 
    FROM (VALUES {selected_skus_tuple}) x(product_id, warehouse_id)
),

selected_prods_with_brand_cat AS (
    SELECT DISTINCT sp.warehouse_id, c.id AS cat_id, b.id AS brand_id, sp.product_id
    FROM selected_prods sp
    JOIN products p ON p.id = sp.product_id 
    JOIN brands b ON b.id = p.brand_id 
    JOIN categories c ON c.id = p.category_id
),

in_stock_retailers AS (
    SELECT DISTINCT retailer_id 
    FROM sales_orders 
    WHERE sales_order_status_id = 6 
        AND channel IN ('retailer', 'telesales')
        AND created_at::date >= DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - INTERVAL '6 months')
),

brand_views AS (
    SELECT 
        vb.retailer_id,
        vb.brand_id,
        vb.category_id,
        vb.brand_name,
        c.name_ar AS cat_name,
        COUNT(DISTINCT vb.event_date) AS view_days,
        MAX(vb.event_date) AS last_view_date
    FROM maxab_events.view_brand vb
    JOIN categories c ON c.id = vb.category_id
    JOIN in_stock_retailers isr ON isr.retailer_id = vb.retailer_id
    WHERE vb.event_timestamp::date BETWEEN CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 10 
          AND CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 2
        AND vb.country LIKE '%Egypt%'
        AND vb.user_id LIKE '%EG_retailers_%'
        AND vb.brand_id <> 'null'
        AND EXISTS (
            SELECT 1 
            FROM sales_orders so
            JOIN PRODUCT_SALES_ORDER pso ON pso.sales_order_id = so.id 
            JOIN products p ON p.id = pso.product_id
            WHERE p.brand_id = vb.brand_id 
                AND p.category_id = vb.category_id
                AND so.created_at::date >= CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 120
                AND so.sales_order_status_id NOT IN (7, 12)
        )
    GROUP BY vb.retailer_id, vb.brand_id, vb.category_id, vb.brand_name, c.name_ar
    HAVING COUNT(DISTINCT vb.event_date) > 1
),

ordered_after_view AS (
    SELECT DISTINCT so.retailer_id, p.brand_id, p.category_id
    FROM sales_orders so
    JOIN PRODUCT_SALES_ORDER pso ON pso.sales_order_id = so.id 
    JOIN products p ON p.id = pso.product_id
    WHERE so.created_at::date >= CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 10
        AND so.sales_order_status_id NOT IN (7, 12)
        AND EXISTS (
            SELECT 1 FROM brand_views bv 
            WHERE bv.retailer_id = so.retailer_id 
                AND bv.brand_id = p.brand_id 
                AND bv.category_id = p.category_id
                AND so.created_at::date >= bv.last_view_date
        )
)

SELECT DISTINCT 
    bv.retailer_id, 
    sp.product_id, 
    sp.warehouse_id, 
    bv.view_days
FROM brand_views bv
JOIN selected_prods_with_brand_cat sp ON sp.brand_id = bv.brand_id AND sp.cat_id = bv.category_id
JOIN materialized_views.retailer_polygon rp ON rp.retailer_id = bv.retailer_id
JOIN DISPATCHING_POLYGONS dp ON dp.district_id = rp.district_id
JOIN WAREHOUSE_DISPATCHING_RULES wdr ON wdr.DISPATCHING_POLYGON_ID = dp.id 
    AND wdr.product_id = sp.product_id 
    AND wdr.warehouse_id = sp.warehouse_id
WHERE NOT EXISTS (
    SELECT 1 FROM ordered_after_view oav 
    WHERE oav.retailer_id = bv.retailer_id 
        AND oav.brand_id = bv.brand_id 
        AND oav.category_id = bv.category_id
)
ORDER BY bv.view_days DESC
    '''
    print("  Fetching view-no-orders retailers...")
    df = query_snowflake(query)
    print(f"    Found {len(df)} view-no-orders retailer-product combinations")
    return df


def get_excluded_retailers() -> pd.DataFrame:
    """
    Get retailers to exclude from SKU discounts.
    Excludes:
    - Retailers with failed last orders
    - Inactive retailers
    - Wholesale tagged retailers
    - Retailers already having active SKU discounts
    
    Returns:
        DataFrame with retailer_id column
    """
    query = f'''
    SELECT retailer_id
    FROM (
        SELECT DISTINCT
            retailer_id,
            sales_order_status_id,
            created_at::date as o_date,
            max(o_date) OVER(PARTITION BY retailer_id) as last_order
        FROM sales_orders so 
        WHERE so.created_at::date >= CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 120
            AND so.sales_order_status_id NOT IN (7, 12)
            AND so.channel IN ('telesales', 'retailer')
        QUALIFY o_date = last_order
    )
    WHERE sales_order_status_id NOT IN (6, 9, 12)
    
    UNION ALL 
    
    SELECT id as retailer_id 
    FROM retailers 
    WHERE activation = 'false'
    
    UNION ALL 
    
    SELECT DISTINCT dta.TAGGABLE_ID as retailer_id
    FROM DYNAMIC_TAGS dt 
    JOIN dynamic_taggables dta ON dt.id = dta.dynamic_tag_id 
    WHERE name LIKE '%whole_sale%'
        AND dt.id > 3000
    '''
    print("  Fetching excluded retailers...")
    df = query_snowflake(query)
    df = df.drop_duplicates()
    print(f"    Found {len(df)} retailers to exclude")
    return df


def get_retailers_with_quantity_discount() -> pd.DataFrame:
    """
    Get retailer-product combinations that already have quantity discounts.
    These should be excluded from SKU discounts.
    
    Returns:
        DataFrame with retailer_id, product_id columns
    """
    # First get active quantity discount tags
    query_tags = f'''
    SELECT DISTINCT
        qdv.product_id,
        qd.dynamic_tag_id AS tag_id
    FROM quantity_discounts qd
    JOIN quantity_discount_values qdv ON qd.id = qdv.quantity_discount_id
    WHERE (CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP()) BETWEEN qd.start_at AND qd.end_at)
        OR ((qd.start_at::date = CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE) 
            AND (CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP()) < qd.start_at))
    AND qd.active = TRUE
    '''
    
    print("  Fetching retailers with quantity discounts...")
    df_tags = query_snowflake(query_tags)
    
    if len(df_tags) == 0:
        print("    No active quantity discounts found")
        return pd.DataFrame(columns=['retailer_id', 'product_id'])
    
    # Get retailers for each tag
    tag_list = ','.join([f"({int(t)})" for t in df_tags['tag_id'].unique()])
    
    query_retailers = f'''
    WITH tags AS (
        SELECT * FROM (VALUES {tag_list}) x(dynamic_tag_id)
    )
    SELECT tags.dynamic_tag_id as tag_id, taggable_id as retailer_id
    FROM dynamic_taggables dt  
    JOIN tags ON tags.dynamic_tag_id = dt.dynamic_tag_id
    '''
    df_retailers = query_snowflake(query_retailers)
    
    # Merge to get retailer-product combinations
    df_qd = df_tags.merge(df_retailers, on='tag_id')
    df_qd = df_qd[['retailer_id', 'product_id']].drop_duplicates()
    
    print(f"    Found {len(df_qd)} retailer-product combinations with quantity discounts")
    return df_qd


def get_retailer_main_warehouse() -> pd.DataFrame:
    """
    Get the main warehouse for each retailer based on last order.
    
    Returns:
        DataFrame with retailer_id, warehouse_id, last_wh columns
    """
    query = f'''
    SELECT retailer_id, warehouse_id, 1 as last_wh 
    FROM (
        SELECT DISTINCT
            so.retailer_id,
            pso.warehouse_id,
            so.created_at::date as o_date,
            max(so.created_at::date) OVER(PARTITION BY so.retailer_id) as max_date
        FROM product_sales_order pso
        JOIN sales_orders so ON so.id = pso.sales_order_id
        JOIN products ON products.id = pso.product_id
        JOIN brands ON products.brand_id = brands.id 
        JOIN categories ON products.category_id = categories.id
        JOIN finance.all_cogs f ON f.product_id = pso.product_id
            AND f.from_date::date <= so.created_at::date
            AND f.to_date::date > so.created_at::date
        JOIN product_units ON product_units.id = products.unit_id  
        WHERE so.created_at::date >= CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 365
            AND so.sales_order_status_id NOT IN (7, 12)
            AND so.channel IN ('telesales', 'retailer')
            AND pso.purchased_item_count <> 0
            AND pso.warehouse_id IN (1, 8, 170, 236, 337, 339, 401, 501, 632, 703, 797, 962)
        GROUP BY 1, 2, 3
        QUALIFY o_date = max_date
    )
    '''
    print("  Fetching retailer main warehouses...")
    df = query_snowflake(query)
    print(f"    Found {len(df)} retailer-warehouse mappings")
    return df


print("Retailer Selection Queries defined ✓")
print("  - get_churned_dropped_retailers()")
print("  - get_category_not_product_retailers()")
print("  - get_out_of_cycle_retailers()")
print("  - get_view_no_orders_retailers()")
print("  - get_excluded_retailers()")
print("  - get_retailers_with_quantity_discount()")
print("  - get_retailer_main_warehouse()")
